**Exploratory Data Analysis on Big Bash League**

<img src="https://i.postimg.cc/TPpHz2n4/BBL-image.png" width="900" height="500"><br>


**Dataset Overview:**

This dataset contains BBL (Big Bash League) cricket match data, consisting of two related datasets: matches.csv and deliveries.csv.


*   **matches.csv** contains match-level information, such as teams, venue, toss decision, match winner, result, and player of the match.
*   **deliveries.csv** contains ball-by-ball information, such as batsman, bowler, runs, extras, overs, innings, and wickets.
The two datasets are connected through the match ID, allowing us to combine match-level and ball-level information.

The dataset can be used to analyze team performance, player performance, batting, bowling, toss impact, venues, match results, and scoring patterns.

## What is Exploratory Data Analysis?

Exploratory Data Analysis (EDA) is the process of exploring, investigating, and understanding data using statistical methods and visualizations. The main objective of EDA is to find useful information, trends, relationships, and patterns in the data.

EDA helps us understand the data before performing further analysis. It involves using Python libraries such as **Pandas, NumPy, Matplotlib, and Seaborn** to clean, analyze, and visualize the data.

In this project, I have selected **Big Bash League (BBL) cricket match and ball-by-ball delivery data** to explore and analyze cricket performance. The dataset can be used to understand **player statistics, team performance, match results, venue performance, batting and bowling patterns, and season-wise trends**.

For example, we can find:

* Which players scored the most runs?
* Which bowlers took the most wickets?
* Which venue hosted the highest number of matches?
* Which venues have higher average scores?
* Which teams have performed the best?
* Which players have the best batting or bowling performance?
* How does team performance change across different seasons?
* How does scoring change during different overs?

We will use Python libraries such as **Pandas, Matplotlib, and Seaborn** to perform exploratory data analysis on the BBL dataset.

### Steps we will follow:

1. Load the BBL match and delivery datasets.
2. Prepare and clean the data using Pandas.
3. Explore the data and understand its different columns and values.
4. Analyze player, team, venue, and match statistics.
5. Create visualizations to identify trends and patterns.
6. Ask and answer interesting questions using the data.
7. Summarize the findings and draw useful conclusions.

By the end of this project, we will gain useful insights into **player performance, team performance, venue statistics, match outcomes, batting and bowling trends, and other important patterns in BBL cricket data**.


---

# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')

### Explanation

*   **pandas**: For data manipulation and analysis.
*   **numpy**: For numerical operations.
*   **matplotlib.pyplot**: For creating static, interactive, and animated visualizations.
*   **seaborn**: For making statistical graphics based on matplotlib.
*   **warnings**: To filter out unwanted warning messages.

Plot configurations are set to ensure visualizations are clear, readable, and consistent throughout the notebook, making it suitable for a professional portfolio.

---

# 2. Load the Dataset

First, we need to extract the `delivers_and_matches.zip` file to access the `deliveries.csv` and `matches.csv` datasets. Then, we will load `deliveries.csv` into a DataFrame called `df` and `matches.csv` into `matches_df`.

In [ ]:
import zipfile

zip_file = "/content/delivers_and_matches.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("data")


In [ ]:
df = pd.read_csv('/content/data/deliveries.csv')
matches_df = pd.read_csv('/content/data/matches.csv')

### Explanation

We have loaded two datasets:

*   `deliveries_df`: Contains `227708` rows and `18` columns. Each row represents a single ball bowled in a match.
*   `matches_df`: Contains `414` rows and `17` columns. Each row represents a complete match.

The `head()` and `tail()` functions provide a quick glimpse of the data, showing the first and last five rows, respectively. The `shape` attribute confirms the number of rows and columns for each DataFrame.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
matches_df.head()

In [ ]:
matches_df.tail()

---

# 3. Understand the Dataset

In [ ]:
display(df.info())



In [ ]:
display(df.dtypes)

In [ ]:
display(matches_df.info())

In [ ]:
display(matches_df.dtypes)

In [ ]:
df.isnull().sum() / len(df) * 100

In [ ]:
merged_df.isnull().sum() / len(merged_df) * 100

### Explanation

#### `deliveries_df`:

*   **Number of rows**: 227708
*   **Number of columns**: 18
*   **Numerical columns**: `id`, `inning`, `over`, `ball`, `batsman_runs`, `extra_runs`, `total_runs`, `non_boundary`, `is_wicket`. These are mostly `int64` type.
*   **Categorical columns**: `batsman`, `non_striker`, `bowler`, `dismissal_kind`, `player_dismissed`, `fielder`, `extras_type`, `batting_team`, `bowling_team`. These are `object` type.
*   **Possible date columns**: None directly in this DataFrame.
*   **Boolean/binary columns**: `non_boundary` (0 or 1), `is_wicket` (0 or 1).
*   **Memory usage**: ~31.3+ MB.

#### `matches_df`:

*   **Number of rows**: 414
*   **Number of columns**: 17
*   **Numerical columns**: `id`, `result_margin`. `id` is `int64`, `result_margin` is `float64`.
*   **Categorical columns**: `city`, `date`, `player_of_match`, `venue`, `neutral_venue`, `team1`, `team2`, `toss_winner`, `toss_decision`, `winner`, `result`, `eliminator`, `method`, `umpire1`, `umpire2`. Most are `object` type.
*   **Possible date columns**: `date` (currently `object`, will need conversion).
*   **Boolean/binary columns**: `neutral_venue` (`N` or `Y`), `eliminator` (`N` or `Y`).
*   **Memory usage**: ~55.2+ KB.

Both DataFrames seem to have appropriate data types for most columns, but `date` in `matches_df` needs to be converted to a datetime object, and some columns in `deliveries_df` (like `dismissal_kind`, `player_dismissed`, `fielder`, `extras_type`) have many non-null values, suggesting missing data.

# 4. Check Missing Values

### Explanation of Missing Values

#### `deliveries_df`:

*   **`fielder` (96.32%)**: This column contains the name of the fielder who caused a run out or stumping, or took a catch. It's naturally missing for the vast majority of deliveries where no such event occurs. We should fill this with a placeholder like 'No Fielder' or 'Not Applicable' to represent the absence of a fielding event.
*   **`player_dismissed` (94.77%)**: This column stores the name of the player who was dismissed. It's `NaN` when no wicket falls on a delivery. Similar to `fielder`, it should be filled with 'No Player Dismissed' or 'Not Applicable'.
*   **`dismissal_kind` (94.77%)**: Describes the type of dismissal (e.g., 'caught', 'bowled'). It's `NaN` when no wicket falls. This should also be filled with 'No Dismissal' or 'Not Applicable'.
*   **`extras_type` (94.92%)**: Indicates the type of extra run (e.g., 'wides', 'noballs'). It's `NaN` when no extra runs are conceded. This should be filled with 'No Extra' or 'Not Applicable'.
*   **`bowling_team` (0.08%)**: This column should always have a value. The small percentage of missing values suggests data entry issues. Since `bowling_team` is a crucial categorical feature, and the percentage is very low, it's best to **remove the rows** where this is missing to maintain data integrity.

#### `matches_df`:

*   **`method` (96.14%)**: This column indicates if a special method (like Duckworth-Lewis-Stern, DLS) was applied to determine the match result due to interruptions. It's naturally `NaN` for most matches that conclude normally. We should fill this with 'Normal' to indicate a standard match conclusion.
*   **`result_margin` (1.69%)**: The margin of victory/defeat. This is `NaN` for tied matches or matches with no result. We should fill this with `0` for ties/no results.
*   **`player_of_match` (0.58%)**: The player who was awarded man of the match. It can be `NaN` if no award was given or the match had no result. We can fill this with 'No Award'.
*   **`winner` (0.58%)**: The winning team. `NaN` for matches with no result or ties. We can fill this with 'No Result'.
*   **`result` (0.58%)**: The match result (e.g., 'runs', 'wickets'). `NaN` for matches with no result or ties. We can fill this with 'No Result'.
*   **`eliminator` (0.58%)**: Indicates if an eliminator (super over) was played. It's `NaN` when no eliminator occurred. We can fill this with 'N' (No).

**Strategy**: We will first merge the two dataframes to get a consolidated view, and then apply these handling strategies to the combined dataset.

---

### Merge `deliveries_df` and `matches_df`

We will merge the two DataFrames on the `id` column, which represents the match ID, to get a consolidated dataset for analysis. We'll use a `left` merge to keep all delivery information and add corresponding match details.

In [ ]:
merged_df = pd.merge(df, matches_df, on='id', how='left')

print('First 5 rows of merged_df:')
display(merged_df.head())

print('\nShape of merged_df (rows, columns):')
display(merged_df.shape)

### Explanation

The `merged_df` now contains `227708` rows and `34` columns. Each row still represents a single delivery, but it now includes all the match-level information from `matches_df` for that specific match `id`.

### Re-check Missing Values in `merged_df`

Now that the datasets are merged, let's re-evaluate the missing values in the comprehensive `merged_df` before proceeding with filling and dropping.

In [ ]:
print('Missing values in merged_df:')
merged_missing = merged_df.isnull().sum()
merged_missing_percent = (merged_df.isnull().sum() / len(merged_df) * 100).sort_values(ascending=False)

merged_missing_summary = pd.DataFrame({
    'Column Name': merged_missing.index,
    'Missing Count': merged_missing.values,
    'Missing Percentage': merged_missing_percent.values
})
display(merged_missing_summary[merged_missing_summary['Missing Count'] > 0])

### Strategy for Handling Missing Values in `merged_df`

Based on the analysis of individual DataFrames and now the merged one, we will apply the following strategies:

1.  **`fielder`, `player_dismissed`, `dismissal_kind`, `extras_type`**: These relate to specific events that do not occur on every ball. They will be filled with 'No Event' or 'Not Applicable' to make them categorical.
2.  **`method`**: This indicates if a special match rule was applied. Missing values mean a 'Normal' match. We will fill with 'Normal'.
3.  **`result_margin`**: The margin of victory. `NaN` for ties or no results. We will fill with `0`.
4.  **`player_of_match`**: Missing if no award or no result. We will fill with 'No Award'.
5.  **`winner`, `result`**: Missing for no result or ties. We will fill with 'No Result'.
6.  **`eliminator`**: Indicates if an eliminator was played. Missing means 'N' (No). We will fill with 'N'.
7.  **`bowling_team`**: A small percentage (0.08%) of missing values in a critical column. We will **drop** these rows as they are few and represent invalid records for analysis.

In [ ]:
# 1. Fill NaN values in event-specific columns with a placeholder
merged_df['fielder'].fillna('No Fielder', inplace=True)
merged_df['player_dismissed'].fillna('No Player Dismissed', inplace=True)
merged_df['dismissal_kind'].fillna('No Dismissal', inplace=True)
merged_df['extras_type'].fillna('No Extra', inplace=True)

# 2. Fill 'method' with 'Normal'
merged_df['method'].fillna('Normal', inplace=True)

# 3. Fill 'result_margin' with 0
merged_df['result_margin'].fillna(0, inplace=True)

# 4. Fill 'player_of_match' with 'No Award'
merged_df['player_of_match'].fillna('No Award', inplace=True)

# 5. Fill 'winner' and 'result' with 'No Result'
merged_df['winner'].fillna('No Result', inplace=True)
merged_df['result'].fillna('No Result', inplace=True)

# 6. Fill 'eliminator' with 'N'
merged_df['eliminator'].fillna('N', inplace=True)

# 7. Drop rows where 'bowling_team' is missing
initial_rows = merged_df.shape[0]
merged_df.dropna(subset=['bowling_team'], inplace=True)
dropped_rows = initial_rows - merged_df.shape[0]


In [ ]:
display(merged_df.isnull().sum())

In [ ]:
display(merged_df.isnull().sum() / len(merged_df) * 100)

### Explanation

All specified missing values have been successfully handled:

*   Event-specific columns (`fielder`, `player_dismissed`, `dismissal_kind`, `extras_type`) were filled with descriptive placeholders.
*   Match outcome related columns (`method`, `result_margin`, `player_of_match`, `winner`, `result`, `eliminator`) were filled with contextually appropriate values.
*   A small number of rows with missing `bowling_team` values were dropped, as these were critical for analysis and few in number. The remaining missing values are 0% for all columns, confirming successful handling.

---

# 5. Data Type Validation

The `date` column in `merged_df` is currently of `object` type, but it represents dates. Converting it to a datetime object will allow for time-series analysis and proper date-based filtering or grouping. We will also inspect the data types of all columns to ensure they are appropriate.

In [ ]:
merged_df['date'] = pd.to_datetime(merged_df['date'])
display(merged_df.info())

In [ ]:
merged_df.describe()

### Explanation

The `date` column has been successfully converted from `object` to `datetime64[ns]` type. This allows for easier manipulation and analysis of match dates. All other columns retain their appropriate data types as determined during initial inspection.

---

# 6. Handle Duplicate Records

It's important to check for and remove any fully duplicate rows that might exist in the dataset to ensure each record is unique and does not skew our analysis. We will check the number of duplicates and then remove them if present.

In [ ]:
merged_df.duplicated().sum()

if merged_df.duplicated().sum() > 0:
    merged_df.drop_duplicates(inplace=True)
    merged_df.duplicated().sum()

else:
    print('No duplicate rows found.')


In [ ]:
display(merged_df.shape)

### Explanation

After checking, no duplicate rows were found in the `merged_df`. This indicates that each delivery record is unique. If duplicates were present, they would have been removed using `drop_duplicates(inplace=True)`, ensuring data integrity for subsequent analysis.

### Handle missing values in the merged DataFrame

Based on our earlier inspection of `df` and the new `matches_df`, we can see some columns with missing values. Let's re-examine the missing values in the `merged_df` and decide on an appropriate strategy.

In [ ]:
merged_df['year'] = merged_df['date'].dt.year

matches_per_year = merged_df.groupby('year')['id'].nunique()

plt.figure(figsize=(10,5))
matches_per_year.plot(kind='bar')
plt.title("Number of Matches Per Year")
plt.xlabel("Year")
plt.ylabel("Number of Matches")
plt.show()

In [ ]:
pom = merged_df.drop_duplicates('id')['player_of_match'].value_counts().head(10)

plt.figure(figsize=(10,6))

pom.sort_values().plot(kind='barh')

plt.title("Top 10 Players by Player of the Match Awards")
plt.xlabel("Number of Awards")
plt.ylabel("Player")

plt.show()

In [ ]:
toss = merged_df.drop_duplicates('id')['toss_decision'].value_counts()

plt.figure(figsize=(6,5))
plt.pie(toss, labels=toss.index, autopct='%1.1f%%')
plt.title("Toss Decision Distribution")
plt.show()

In [ ]:
matches = merged_df.drop_duplicates('id')

matches['toss_match_winner'] = (
    matches['toss_winner'] == matches['winner']
)

result = matches['toss_match_winner'].value_counts()

plt.figure(figsize=(6,5))

plt.pie(
    result,
    labels=['Toss Winner Lost', 'Toss Winner Won'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.4)
)

plt.title("Does Winning the Toss Lead to Winning the Match?")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df['total_runs'], bins=10)

plt.title("Distribution of Runs per Ball")
plt.xlabel("Runs")
plt.ylabel("Number of Balls")
plt.show()

In [ ]:
top_batsmen = df.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
top_batsmen.plot(kind='bar')

plt.title("Top 10 Run Scorers")
plt.xlabel("Batsman")
plt.ylabel("Runs")
plt.xticks(rotation=45)
plt.show()

In [ ]:
wickets = df[df['is_wicket'] == 1]

top_bowlers = wickets.groupby('bowler').size().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
top_bowlers.plot(kind='bar')

plt.title("Top 10 Wicket Taking Bowlers")
plt.xlabel("Bowler")
plt.ylabel("Wickets")
plt.xticks(rotation=45)
plt.show()

In [ ]:
dismissals = df[df['dismissal_kind'].notna()]

plt.figure(figsize=(10,6))

sns.countplot(
    data=dismissals,
    y='dismissal_kind',
    order=dismissals['dismissal_kind'].value_counts().index
)

plt.title("Types of Dismissals")
plt.xlabel("Number of Dismissals")
plt.ylabel("Dismissal Type")
plt.show()

In [ ]:
runs_by_over = df.groupby('over')['total_runs'].sum()

plt.figure(figsize=(12,5))

runs_by_over.plot(kind='line', marker='o')

plt.title("Total Runs Scored by Over")
plt.xlabel("Over")
plt.ylabel("Runs")
plt.grid()
plt.show()

In [ ]:
team_data = pd.crosstab(
    df['batting_team'],
    df['bowling_team']
)

plt.figure(figsize=(12,8))

sns.heatmap(team_data, annot=True, fmt='d')

plt.title("Batting Team vs Bowling Team")
plt.xlabel("Bowling Team")
plt.ylabel("Batting Team")

plt.show()

In [ ]:
extras = df.groupby('extras_type')['extra_runs'].sum().sort_values(ascending=False)

plt.figure(figsize=(8,5))

extras.plot(kind='bar')

plt.title("Extra Runs by Type")
plt.xlabel("Extra Type")
plt.ylabel("Runs")
plt.xticks(rotation=45)

plt.show()

In [ ]:
pom = merged_df.drop_duplicates('id')['player_of_match'].value_counts().head(10)
pom = pom.sort_values(ascending=False) # Sort in descending order for better visualization

plt.figure(figsize=(10,6))

sns.barplot(x=pom.values, y=pom.index, palette='viridis')

plt.title("Top 10 Players by Player of the Match Awards")
plt.xlabel("Number of Awards")
plt.ylabel("Player")

plt.show()

In [ ]:
merged_df['is_boundary'] = merged_df['batsman_runs'].isin([4, 6])

**Strike Rate** 🏏

Strike Rate = (Runs / Balls Faced) × 100

In [ ]:
player_stats = merged_df.groupby('batsman').agg(
    runs=('batsman_runs', 'sum'),
    balls=('batsman_runs', 'count')
)

player_stats['strike_rate'] = (
    player_stats['runs'] / player_stats['balls']
) * 100

player_stats = player_stats.sort_values(
    'strike_rate',
    ascending=False
)

# Top 10 players
top10 = player_stats.head(10)

# Lollipop chart
plt.figure(figsize=(12, 6))

plt.stem(
    top10.index,
    top10['strike_rate']
)

plt.xlabel('Player')
plt.ylabel('Strike Rate')
plt.title('Top 10 Players by Strike Rate')

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

**Economy Rate 🎯**

Economy Rate = Runs Conceded / Overs Bowled

In [ ]:
bowler_stats = merged_df.groupby('bowler').agg(
    runs_conceded=('total_runs', 'sum'),
    balls=('ball', 'count')
)

bowler_stats['overs'] = bowler_stats['balls'] / 6

bowler_stats['economy_rate'] = (
    bowler_stats['runs_conceded'] / bowler_stats['overs']
)

# Top 10 best economy rates
top10 = bowler_stats.sort_values(
    'economy_rate'
).head(10).copy()

# Round to 2 decimals
top10['economy_rate'] = top10['economy_rate'].round(2)

# Reverse order for graph
top10 = top10.sort_values('economy_rate')

# Create graph
plt.figure(figsize=(12, 7))

plt.barh(
    top10.index,
    top10['economy_rate']
)

# Add values
for bowler, economy in zip(
    top10.index,
    top10['economy_rate']
):
    plt.text(
        economy + 0.05,
        bowler,
        f'{economy:.2f}',
        va='center'
    )

plt.xlabel('Economy Rate')
plt.ylabel('Bowler')
plt.title('Top 10 Bowlers by Economy Rate')

plt.tight_layout()
plt.show()

**Boundary Count 💥**

We Count the number of 4s and 6s

In [ ]:
boundaries = merged_df[
    merged_df['batsman_runs'].isin([4, 6])
]

boundary_count = boundaries.groupby(
    ['batsman', 'batsman_runs']
).size().unstack(fill_value=0)

boundary_count = boundary_count.rename(
    columns={4: 'Fours', 6: 'Sixes'}
)

boundary_count['Total_Boundaries'] = (
    boundary_count['Fours'] + boundary_count['Sixes']
)

boundary_count = boundary_count.sort_values(
    'Total_Boundaries',
    ascending=False
)

print(boundary_count.head(10))
top10 = boundary_count.head(10)

plt.figure(figsize=(12, 7))

plt.bar(
    top10.index,
    top10['Fours'],
    label='Fours'
)

plt.bar(
    top10.index,
    top10['Sixes'],
    bottom=top10['Fours'],
    label='Sixes'
)

plt.xlabel('Player')
plt.ylabel('Number of Boundaries')
plt.title('Top 10 Players - Fours and Sixes')

plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

**Dot Balls ⚫**

A dot ball means 0 runs were scored from the delivery.

In [ ]:
import matplotlib.pyplot as plt

dot_balls = merged_df[
    merged_df['total_runs'] == 0
]

dot_ball_count = dot_balls.groupby('bowler').size()

dot_ball_count = dot_ball_count.sort_values(
    ascending=False
)

print(dot_ball_count.head(10))

top10 = dot_ball_count.head(10)

plt.figure(figsize=(12, 7))

plt.barh(
    top10.index[::-1],
    top10.values[::-1]
)

# Add values at the end of bars
for bowler, value in zip(
    top10.index[::-1],
    top10.values[::-1]
):
    plt.text(
        value + 2,
        bowler,
        str(value),
        va='center'
    )

plt.xlabel('Number of Dot Balls')
plt.ylabel('Bowler')
plt.title('Top 10 Bowlers by Dot Balls')

plt.tight_layout()
plt.show()

**Powerplay Runs** ⚡

In BBL, the Powerplay is generally overs 0–5.

Your over column starts from 0, so:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

powerplay = merged_df[
    merged_df['over'] < 6
]

powerplay_runs = powerplay.groupby(
    ['batsman']
)['total_runs'].sum()

powerplay_runs = powerplay_runs.sort_values(
    ascending=False
)

print(powerplay_runs)

# ---------------- KDE PLOT ----------------

plt.figure(figsize=(12, 6))

sns.kdeplot(
    powerplay_runs,
    fill=True
)

plt.xlabel('Powerplay Runs')
plt.ylabel('Density')
plt.title('Distribution of Powerplay Runs by Batters')

plt.tight_layout()
plt.show()

**BY TEAM**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

powerplay = merged_df[
    merged_df['over'] < 6
]

powerplay_runs = powerplay.groupby(
    ['batting_team']
)['total_runs'].sum()

print(powerplay_runs.head(10))


# ---------------- HEATMAP ----------------

heatmap_data = powerplay_runs.to_frame(
    name='Powerplay Runs'
)

plt.figure(figsize=(8, 10))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    linewidths=1,
    cbar_kws={'label': 'Runs'}
)

plt.title('Powerplay Runs by Team')
plt.xlabel('')
plt.ylabel('Team')

plt.tight_layout()
plt.show()

**Death-Over Runs**🔥

For BBL, Death Overs are usually overs 16–19

In [ ]:
death_overs = merged_df[

    merged_df['over'] >= 16

]

death_runs = death_overs.groupby(

    ['batsman']

)['total_runs'].sum()

death_runs = death_runs.sort_values(

    ascending=False

)

print(death_runs.head(10))


# ---------------- BUBBLE CHART ----------------

import matplotlib.pyplot as plt

top10 = death_runs.head(10)

plt.figure(figsize=(12, 7))

plt.scatter(
    range(len(top10)),
    top10.values,
    s=top10.values * 8,
    alpha=0.7
)

for i, (player, runs) in enumerate(top10.items()):
    plt.text(
        i,
        runs + 5,
        f'{player}\n{runs}',
        ha='center',
        fontsize=10
    )

plt.xticks(
    range(len(top10)),
    top10.index,
    rotation=45,
    ha='right'
)

plt.xlabel('Batter')
plt.ylabel('Death Over Runs')
plt.title('Top 10 Batters by Death Over Runs')

plt.tight_layout()
plt.show()

**BY TEAM**

In [ ]:
death_overs = merged_df[

    merged_df['over'] >= 16

]

death_runs = death_overs.groupby(

    ['batting_team']

)['total_runs'].sum()

print(death_runs.head(10))


# ---------------- DOT PLOT ----------------

import matplotlib.pyplot as plt

death_runs = death_runs.sort_values(ascending=True)

plt.figure(figsize=(12, 7))

plt.plot(
    death_runs.values,
    death_runs.index,
    'o',
    markersize=10
)

for team, runs in zip(
    death_runs.index,
    death_runs.values
):
    plt.text(
        runs + 5,
        team,
        str(runs),
        va='center'
    )

plt.xlabel('Death Over Runs')
plt.ylabel('Batting Team')
plt.title('Death Over Runs by Team')

plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

**Player Average**🏏

Batting Average = Total Runs / Number of Times Dismissed

In [ ]:
player_runs = merged_df.groupby(

    'batsman'

)['batsman_runs'].sum()

dismissals = merged_df[

    merged_df['player_dismissed'].notna()

].groupby('player_dismissed').size()

player_average = (

    player_runs / dismissals

)

player_average = player_average.round(2)

player_average = player_average.sort_values(

    ascending=False

)

print(player_average.head(10))


# ---------------- LOLLIPOP CHART ----------------

import matplotlib.pyplot as plt

top10 = player_average.head(10).sort_values()

plt.figure(figsize=(12, 7))

plt.vlines(
    x=range(len(top10)),
    ymin=0,
    ymax=top10.values,
    linewidth=2
)

plt.scatter(
    range(len(top10)),
    top10.values,
    s=120
)

for i, value in enumerate(top10.values):
    plt.text(
        i,
        value + 1,
        f'{value:.2f}',
        ha='center'
    )

plt.xticks(
    range(len(top10)),
    top10.index,
    rotation=45,
    ha='right'
)

plt.xlabel('Player')
plt.ylabel('Batting Average')
plt.title('Top 10 Players by Batting Average')

plt.tight_layout()
plt.show()

**Team Scoring Rate** 📈

A simple scoring rate is:Total Runs / Overs

In [ ]:
team_scoring_rate = (

    (team_runs / team_overs)

    .round(2)

    .sort_values(ascending=False)

)

print(team_scoring_rate)


# ---------------- STEP PLOT ----------------

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

plt.step(
    team_scoring_rate.index,
    team_scoring_rate.values,
    where='mid',
    marker='o'
)

for i, value in enumerate(team_scoring_rate.values):
    plt.text(
        i,
        value + 0.05,
        f'{value:.2f}',
        ha='center'
    )

plt.xlabel('Team')
plt.ylabel('Scoring Rate')
plt.title('Team Scoring Rate')

plt.xticks(rotation=45)

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.4
)

plt.tight_layout()
plt.show()

## Summary:

In the given BBL dataset, I explored and learned many things like,

* How to clean and prepare the dataset for analysis.

* How to use Pandas and NumPy for data manipulation, grouping, filtering, and calculations.

* In the process of analyzing the dataset, I explored player and team performance using different conditions and calculations.

* I analyzed important factors such as **runs, boundaries, batting average, strike rate, Powerplay runs, death-over runs, scoring rate, economy rate, and dot balls**.

* I used **Seaborn and Matplotlib** libraries to visualize the data and understand the patterns and performance of players and teams.

* By performing this EDA, I identified important insights about **top-performing players, team scoring patterns, batting performance, and bowling performance**, which can be useful for further BBL analysis and predictive modeling.
